In [ ]:
# This script is demonstrating practical implementation of the LazySlide Protocol - Feature Extraction. It includes preprocessing steps as outlined here: 
#https://lazyslide.readthedocs.io/en/v0.7.2/tutorials/01_preprocessing.html#

# To begin
    #1. source /mnt/data0/ikoluk/miniconda3/bin/activate
    #2. conda activate wsi_env1
    #3. cd /mnt/data0/ikoluk/lazyslide
    #4. jupyter lab --no-browser --port=8888

In [ ]:
# Importing lazyslide libaries and sample data

!pip install scanpy anndata pooch matplotlib numpy marsilea # one time

from wsidata import open_wsi
import lazyslide as zs
import matplotlib.pyplot as plt
import scanpy as sc
import pooch
import numpy as np
import marsilea as ma
import marsilea.plotter as mp
from scipy.stats import zscore
import pandas as pd
import os

slide_path = "/mnt/data0/ikoluk/wsi_project/data/GTEX-11DXX-1626.svs"
store_path = "/mnt/data0/ikoluk/wsi_project/data/GTEX-11DXX-1626.zarr"

wsi = open_wsi(slide_path, store=store_path)

In [ ]:
# Run once, feature extraction is saved on WSI object
HF_token = os.environ.get("HF_TOKEN", "hf_xxxxxxxxxxxxxxxxx")  
zs.pp.find_tissues(wsi)
zs.pp.tile_tissues(wsi, 64)
zs.tl.feature_extraction(wsi, "h0-mini", token="hf_xxxxxxxxxxxxxxxxx")
zs.tl.feature_aggregation(wsi, feature_key="h0-mini")
wsi.write()

In [ ]:
# Exploring tile-level embeddings
adata = wsi["h0-mini_tiles"]
sc.pp.scale(adata)
sc.pp.pca(adata)
sc.pp.neighbors(adata)
sc.tl.umap(adata)
sc.pl.umap(adata, color = "leiden")
sc.tl.leiden(adata, resolution=0.2, key_added="leiden")

In [ ]:
# Creating a feature list based on strength of embeddings
sc.tl.rank_genes_groups(adata, groupby="leiden")
features = set()
for i in adata.obs["leiden"].unique():
    names = sc.get.rank_genes_groups_df(adata, i).names
    features.update(list(names[0:10]) + list(names[-10:]))
features = list(features)

In [ ]:
key = "leiden"
h = ma.Heatmap(zscore(adata[:, features].X.T), height=2, width=4, label="Feature")
order = sorted(adata.obs[key].unique())
h.group_cols(adata.obs[key], order=order)
h.add_top(mp.Chunk(order, fill_colors=adata.uns[f"{key}_colors"], padding=2), pad=0.05)
h.add_dendrogram("right", method="average", linewidth=0.1)
h.add_legends()
h.render()

In [ ]:
# Identifying spatial domains
key = "leiden"
clusters = adata.obs[key].cat.categories  
colors = adata.uns[f"{key}_colors"]       
palette_dict = dict(zip(clusters, colors))

zs.pl.tiles(
    wsi,
    feature_key="h0-mini",
    color="leiden",
    alpha=0.5,
    palette=palette_dict,
    show_contours=False,
)

In [ ]:
# Spacial domains in UTAG
zs.pp.tile_graph(wsi)
zs.tl.spatial_features(wsi, feature_key="h0-mini")

zs.tl.spatial_domain(
    wsi,
    feature_key="h0-mini",
    layer="spatial_features",   # ← this matches what spatial_features created
    resolution=0.3
)
zs.pl.tiles(wsi, color="domain", alpha=0.5)

In [ ]:
# Text embeddings
zs.pp.tile_tissues(wsi, 128, mpp=0.5, key_added="text_tiles")
zs.tl.feature_extraction(wsi, "plip", tile_key="text_tiles")
terms = ["mucosa", "submucosa", "musclaris", "lymphocyte"]

embeddings = zs.tl.text_embedding(terms, model="plip")
zs.tl.text_image_similarity(wsi, embeddings, model="plip", tile_key="text_tiles")
zs.pl.tiles(
    wsi,
    feature_key="plip_text_tiles_text_similarity",
    tile_key="text_tiles",
    color=terms,
    cmap="rainbow",
    show_image=False,
    tissue_id=3,
    ncols=2,
    alpha=0.7,
)

In [ ]:
# FIGURE 1: Raw WSI Image
import numpy as np
import matplotlib.pyplot as plt
from wsidata import open_wsi

wsi = open_wsi("data/GTEX-11DXX-1626.svs", store=None)
thumb_da = wsi.images["wsi_thumbnail"]    # xarray.DataArray (3, H, W)
thumb = thumb_da.values                   # numpy (3, H, W)
thumb = np.transpose(thumb, (1, 2, 0))
height, width = thumb.shape[:2]
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
ax.imshow(thumb, origin="upper")
ax.set_xlim(0, width)
ax.set_ylim(height, 0)           
ax.set_xlabel(f"X (pixels): 0 → {width}")
ax.set_ylabel(f"Y (pixels): 0 → {height}")

ax.text(
    0.99, 0.01,
    f"{width} × {height} px",
    transform=ax.transAxes,
    ha="right", va="bottom",
    color="white",
    fontsize=12,
    backgroundcolor="black",
    alpha=0.6
)

In [ ]:
# FIGURE 2: SEGMENTED TISSUE0 WITH LAZYSLIDE
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
zs.pl.tissue(wsi, tissue_id=0, ax=ax)   # draws the same thumbnail
ax.set_extent(extent)                   # switch to µm coords
ax.set_xlabel("µm"); ax.set_ylabel("µm")

In [ ]:
# FIGURE 2: SEGMENTED TISSUE0 WITH DL
import numpy as np
import geopandas as gpd
import matplotlib.pyplot as plt
from wsidata import open_wsi
import lazyslide as zs

wsi = open_wsi("data/GTEX-11DXX-1626.svs",
               store="data/GTEX-11DXX-1626.zarr")

# redo DL segmentation if needed
zs.pp.find_tissues(wsi)
zs.seg.tissue(wsi, key_added="dl-tissues")

# pull out thumbnail array & reorder to H×W×3
thumb = wsi.images["wsi_thumbnail"].values.transpose(1,2,0)

# build a GeoDataFrame of your DL polygons
gdf = wsi.shapes["dl-tissues"]  # this is a GeoDataFrame

# plot
fig, ax = plt.subplots(figsize=(8,8), dpi=200)
ax.imshow(thumb, origin="upper")
ax.axis("off")

# overlay the outlines
gdf.geometry.plot(
    ax=ax,
    facecolor="none",
    edgecolor="red",
    linewidth=1.5,
    alpha=0.6
)

In [ ]:
import matplotlib.pyplot as plt
from wsidata import open_wsi
import lazyslide as zs

# 1) open the slide (re-using your existing Zarr store so we don't re-tile)
wsi = open_wsi("data/GTEX-11DXX-1626.svs", store="data/GTEX-11DXX-1626.zarr")

# 2) detect tissues (creates wsi.shapes["tissues"])
zs.pp.find_tissues(wsi)

# 3) score each tissue for brightness and redness
zs.pp.score_tissues(wsi, scorers=["brightness", "redness"])

# 4) if you want geometric props too (optional)
zs.tl.tissue_props(wsi)

# 5) inspect what shape‐layers you now have
print("Available shape layers:", list(wsi.shapes.keys()))
# ➜ should include "tissues"

# 6) plot: draw the thumbnail, then overlay the tissues colored by brightness
fig, ax = plt.subplots(figsize=(8,8), dpi=300)
zs.pl.tissue(wsi, ax=ax)        # base thumbnail
gdf = wsi.shapes["tissues"]                       # now exists
# use geopandas built‐in plot to overlay
gdf.plot(
    column="brightness",
    cmap="magma",
    linewidth=1.0,
    edgecolor="black",
    alpha=0.7,
    ax=ax
)

In [ ]:
import matplotlib.pyplot as plt

zs.pp.tile_tissues(wsi, 256, mpp=0.5)
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
zs.pl.tiles(
    wsi,
    tissue_id=0,
    linewidth=0.5,
    ax=ax
)


In [ ]:
# QUALITY ASSESSMENT
zs.pp.score_tiles(wsi, scorers=["contrast", "focus"])
fig, ax = plt.subplots(figsize=(8, 8), dpi=300)
zs.pl.tiles(
    wsi,
    tissue_id=0,
    color="contrast",
    cmap="rainbow",
    ax=ax
)

In [ ]:
tiles = wsi.shapes["tiles"]
wsi.shapes["tiles"] = tiles.iloc[:2000]

# run a fast extraction on those 2k tiles
zs.tl.feature_extraction(
    wsi,
    model="h0-mini",
    amp=False,           # no autocast
    batch_size=256,
    num_workers=4,
)
features = zs.tl.feature_extraction(
    wsi,
    model="h0-mini",
    amp=False,
    return_features=True,   # ← you get back a NumPy array
)
print("features shape:", features.shape)

In [ ]:
# table of normalized, pooled CNN‐derived embeddings
adata = wsi.fetch.features_anndata("h0-mini")
df = pd.DataFrame(
    adata.X,
    index=adata.obs_names,
    columns=adata.var_names
)
print(df)